In [1]:
!pip install --quiet ultralytics
from ultralytics import YOLO

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 4.2 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.


In [2]:
import os, random, glob
from pathlib import Path
from collections import Counter

root = Path('/kaggle/input/datasets/simuletic/long-distance-wildfire-and-smoke-detection-dataset/Long-Distance  Wildfire & Smoke Dataset')
out = Path('/kaggle/working/dataset')

images_dir = root / 'images'
labels_dir = root / 'labels'

img_files = sorted([f for f in images_dir.iterdir() if f.suffix.lower() in ('.jpg', '.jpeg', '.png')])
random.seed(42)
random.shuffle(img_files)

split_idx = int(len(img_files) * 0.8)
train_imgs, val_imgs = img_files[:split_idx], img_files[split_idx:]

missing_labels = 0
for split_name, files in [('train', train_imgs), ('val', val_imgs)]:
    (out / 'images' / split_name).mkdir(parents=True, exist_ok=True)
    (out / 'labels' / split_name).mkdir(parents=True, exist_ok=True)
    for img_path in files:
        dst_img = out / 'images' / split_name / img_path.name
        if not dst_img.exists():
            os.symlink(img_path, dst_img)
        label_path = labels_dir / (img_path.stem + '.txt')
        dst_label = out / 'labels' / split_name / label_path.name
        if label_path.exists() and not dst_label.exists():
            os.symlink(label_path, dst_label)
        elif not label_path.exists():
            missing_labels += 1

print(f"Train: {len(train_imgs)} images, Val: {len(val_imgs)} images")
print(f"Images with no matching label file: {missing_labels}")

Train: 191 images, Val: 48 images
Images with no matching label file: 0


In [3]:
print(open(root / 'data.yaml').read())

# Simuletic Smoke And Fire Emergency Dataset. Supports YOLOv8 and YOLO11.

path: .          
train: images    
val: images       


nc: 2           
names:
  0: smoke
  1: wildfire



In [4]:
train_counts = Counter()
flame_train_files = []
for f in glob.glob(str(out / 'labels' / 'train' / '*.txt')):
    lines = open(f).read().splitlines()
    classes_in_file = set()
    for line in lines:
        if not line.strip():
            continue
        cls = line.split()[0]
        train_counts[cls] += 1
        classes_in_file.add(cls)
    if '1' in classes_in_file:
        flame_train_files.append(f)

print("Train instance counts by class:", train_counts)
print(f"Train images containing wildfire/flame class: {len(flame_train_files)}")

Train instance counts by class: Counter({'0': 204, '1': 58})
Train images containing wildfire/flame class: 38


In [5]:
OVERSAMPLE_FACTOR = 3
added = 0
for label_path in flame_train_files:
    label_path = Path(label_path)
    stem = label_path.stem
    matches = glob.glob(str(out / 'images' / 'train' / f'{stem}.*'))
    if not matches:
        continue
    img_path = Path(matches[0])
    for i in range(1, OVERSAMPLE_FACTOR):
        new_img = out / 'images' / 'train' / f'{stem}_dup{i}{img_path.suffix}'
        new_label = out / 'labels' / 'train' / f'{stem}_dup{i}.txt'
        if not new_img.exists():
            os.symlink(img_path.resolve(), new_img)
            added += 1
        if not new_label.exists():
            os.symlink(label_path.resolve(), new_label)

print(f"Added {added} duplicated flame-containing images to train split")

Added 76 duplicated flame-containing images to train split


In [6]:
train_counts_after = Counter()
for f in glob.glob(str(out / 'labels' / 'train' / '*.txt')):
    for line in open(f):
        if not line.strip():
            continue
        train_counts_after[line.split()[0]] += 1
print("Train instance counts by class after oversampling:", train_counts_after)

Train instance counts by class after oversampling: Counter({'0': 286, '1': 174})


In [7]:
import yaml

data = {
    'path': str(out),
    'train': 'images/train',
    'val': 'images/val',
    'names': {0: 'smoke', 1: 'wildfire'}
}

with open('/kaggle/working/data.yaml', 'w') as f:
    yaml.dump(data, f, default_flow_style=False, sort_keys=False)

print(open('/kaggle/working/data.yaml').read())

path: /kaggle/working/dataset
train: images/train
val: images/val
names:
  0: smoke
  1: wildfire



In [8]:
model = YOLO("yolo26s.pt")

results = model.train(
    data="/kaggle/working/data.yaml",
    epochs=150,
    imgsz=960,
    device=[0, 1],
    cls=2.0,
    patience=40,
    scale=0.5,
    copy_paste=0.3,
    mixup=0.15,   
)

Ultralytics 8.4.142 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
                                                        CUDA:1 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=2.0, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.3, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=960, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.15, mode=train, model=yolo26s.pt, momentum=0.9

/usr/local/lib/python3.12/dist-packages/ray/train/_internal/session.py:676: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(



      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
      2/150      6.94G      2.206      38.17    0.02093         13        960: 100% ━━━━━━━━━━━━ 17/17 2.6it/s 6.6s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 3.1it/s 0.6s
                   all         48         64      0.106      0.204     0.0381     0.0102

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
      3/150      6.94G      2.084      20.01    0.01794         17        960: 100% ━━━━━━━━━━━━ 17/17 2.7it/s 6.4s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 2.8it/s 0.7s
                   all         48         64      0.113      0.174      0.063      0.014

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
      4/150      6.94G      2.178      14.48    0.01903         14        960: 100% ━━━━━━━━━━━━ 17/17 

In [9]:
import pandas as pd

metrics = model.val(
    data="/kaggle/working/data.yaml",
    split="val",
    conf=0.1,
    verbose=False
)

metrics_data = {
    "Metric": ["Precision", "Recall", "mAP50", "mAP50-95"],
    "Value": [
        f"{metrics.box.mp:.4f}",
        f"{metrics.box.mr:.4f}",
        f"{metrics.box.map50:.4f}",
        f"{metrics.box.map:.4f}"
    ]
}

df = pd.DataFrame(metrics_data)
print("\nValidation Summary Results (overall):\n")
print(df.to_markdown(index=False))

Ultralytics 8.4.142 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO26s summary (fused): 120 layers, 9,465,954 parameters, 0 gradients, 20.8 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 956.8±297.3 MB/s, size: 1721.9 KB)
val: Scanning /kaggle/working/dataset/labels/val.cache... 48 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 48/48 12.6Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 1.1it/s 2.8s
                   all         48         64       0.52      0.547      0.502      0.202
Speed: 10.4ms preprocess, 25.3ms inference, 0.0ms loss, 3.2ms postprocess per image
Results saved to /kaggle/working/runs/detect/val

Validation Summary Results (overall):

| Metric    |   Value |
|:----------|--------:|
| Precision |  0.5196 |
| Recall    |  0.5468 |
| mAP50     |  0.5019 |
| mAP50-95  |  0.202  |


In [10]:
class_names = data['names']
per_class_rows = []
for i, name in class_names.items():
    p, r, ap50, ap = metrics.box.class_result(i)
    per_class_rows.append({"Class": name, "Precision": f"{p:.4f}", "Recall": f"{r:.4f}", "mAP50": f"{ap50:.4f}", "mAP50-95": f"{ap:.4f}"})

df_class = pd.DataFrame(per_class_rows)
print("\nValidation Summary Results (per-class):\n")
print(df_class.to_markdown(index=False))


Validation Summary Results (per-class):

| Class    |   Precision |   Recall |   mAP50 |   mAP50-95 |
|:---------|------------:|---------:|--------:|-----------:|
| smoke    |      0.7796 |   0.8627 |  0.8506 |     0.352  |
| wildfire |      0.2596 |   0.2308 |  0.1532 |     0.0521 |


In [11]:
!zip -r -q archive.zip "/kaggle/working/runs"
print("\nArchive package generated completely at: /kaggle/working/archive.zip")


Archive package generated completely at: /kaggle/working/archive.zip
